# Connect to Docker Environments and WRDS

In [ ]:
import os
import sys
import wrds
import pandas as pd
import getpass
from influxdb_client_3 import InfluxDBClient3, Point, WriteOptions, write_client_options

# 🐳 Docker Environment Detection and Configuration
print("🔍 Detecting environment...")

def is_docker_environment():
    """Check if running inside Docker container"""
    return os.path.exists('/.dockerenv') or os.environ.get('JUPYTER_SERVER_ROOT')

def get_influxdb_config():
    """Get InfluxDB configuration based on environment"""
    in_docker = is_docker_environment()
    
    if in_docker:
        # Docker environment - use container names and environment variables
        config = {
            'host': f"http://{os.environ.get('INFLUXDB_HOST', 'influxdb')}:{os.environ.get('INFLUXDB_PORT', '8086')}",
            'token': os.environ.get('INFLUXDB_TOKEN', 'qtab_factor_token_2024_secure'),
            'database': os.environ.get('INFLUXDB_BUCKET', 'jkp_factors'),
            'org': os.environ.get('INFLUXDB_ORG', 'qtab')
        }
        print(f"🐳 Docker environment detected")
        print(f"📍 InfluxDB Host: {config['host']}")
    else:
        # Local environment - use localhost
        config = {
            'host': "http://localhost:8086",
            'token': "qtab_factor_token_2024_secure",
            'database': "jkp_factors", 
            'org': "qtab"
        }
        print(f"💻 Local environment detected")
        print(f"📍 InfluxDB Host: {config['host']}")
    
    return config

# Get InfluxDB configuration
influx_config = get_influxdb_config()
print(f"🔗 InfluxDB Database: {influx_config['database']}")
print(f"🏢 Organization: {influx_config['org']}")

In [ ]:
# 🔐 WRDS Connection with Environment Variable Support
print("🔐 Establishing WRDS connection...")

def get_wrds_connection():
    """
    Get WRDS connection using environment variables or interactive input
    """
    # Try environment variables first
    username = os.environ.get('WRDS_USERNAME')
    password = os.environ.get('WRDS_PASSWORD')
    
    if username and password:
        print(f"✅ Using WRDS credentials from environment for user: {username}")
        try:
            connection = wrds.Connection(wrds_username=username, wrds_password=password)
            print("🔗 WRDS connection established successfully")
            return connection
        except Exception as e:
            print(f"❌ Failed to connect with environment credentials: {e}")
            print("🔄 Falling back to interactive input...")
    else:
        print("⚠️  No WRDS credentials found in environment variables")
        print("💡 Tip: Create a .env file with WRDS_USERNAME and WRDS_PASSWORD")
    
    # Fall back to interactive input
    print("🔐 Please enter your WRDS credentials:")
    username = input("WRDS Username: ")
    password = getpass.getpass("WRDS Password: ")
    
    try:
        connection = wrds.Connection(wrds_username=username, wrds_password=password)
        print("✅ WRDS connection established successfully")
        return connection
    except Exception as e:
        print(f"❌ Failed to connect to WRDS: {e}")
        raise

# Establish WRDS connection
wrds_db = get_wrds_connection()

# Test the connection
# try:
#     test_result = wrds_db.raw_sql("SELECT 1 as test", obs=1)
#     print(f"🏓 WRDS connection test successful: {len(test_result)} row returned")
# except Exception as e:
#     print(f"⚠️  WRDS connection test failed: {e}")

# Load Data from WRDS

In [ ]:
from datetime import datetime, timedelta
from typing import List, Dict, Optional
import pandas as pd

class JKPDataLoader:
    """
    Modular data loader for JKP (Jensen, Kelly, Pedersen) global characteristics data
    Validated against JKP_Characteristics_README.md and characteristics.csv
    """
    
    # Factor mappings based on JKP documentation and academic validation
    FACTOR_MODELS = {
        'fama_french_5': {
            # Core Fama-French 5 factors - validated against JKP documentation
            'value': ['be_me', 'ni_me', 'ocf_me'],  # Book-to-market (primary), earnings yield, cash flow yield
            'size': ['market_equity'],  # Market equity (primary FF size measure)
            'momentum': ['ret_12_1'],  # 12-1 month momentum (Jegadeesh Titman 1993)
            'profitability': ['gp_at', 'ni_at'],  # Gross profits/assets (Novy-Marx 2013), ROA
            'investment': ['at_gr1', 'oaccruals_at'],  # Asset growth (Cooper et al 2008), Operating accruals (Sloan 1996)
            'market': ['ret_exc', 'beta_60m']  # Market excess returns and beta
        },
        'fama_french_5_extended': {
            # Extended version with more characteristics per factor
            'value': ['be_me', 'ni_me', 'ocf_me', 'sale_me', 'fcf_me'],
            'size': ['market_equity', 'me'],
            'momentum': ['ret_12_1', 'ret_6_1', 'ret_12_7'],
            'profitability': ['gp_at', 'ni_at', 'ni_be', 'ope_be'],
            'investment': ['at_gr1', 'capx_gr1', 'oaccruals_at', 'netis_at'],
            'market': ['ret_exc', 'beta_60m', 'beta_252d']
        },
        'quality_comprehensive': {
            # Comprehensive quality-focused model
            'profitability': ['gp_at', 'ni_at', 'ni_be', 'ope_be', 'ocf_at'],
            'quality_composite': ['f_score', 'qmj', 'qmj_prof', 'qmj_safety'],
            'earnings_quality': ['ni_ar1', 'earnings_variability', 'oaccruals_at'],
            'growth_quality': ['sale_gr1', 'at_gr1', 'qmj_growth'],
            'financial_strength': ['at_be', 'cash_at', 'z_score']
        },
        'trading_frictions': {
            # Trading frictions and liquidity factors
            'size': ['market_equity', 'me'],
            'volatility': ['ivol_capm_252d', 'ivol_ff3_21d', 'rvol_21d'],
            'liquidity': ['ami_126d', 'turnover_126d', 'zero_trades_126d'],
            'beta': ['beta_60m', 'beta_252d', 'betadown_252d'],
            'skewness': ['iskew_capm_21d', 'rskew_21d', 'coskew_21d'],
            'extremes': ['rmax1_21d', 'rmax5_21d', 'rmax5_rvol_21d']
        },
        'academic_research': {
            # Comprehensive academic research model
            'value': ['be_me', 'ni_me', 'fcf_me', 'sale_me'],
            'momentum': ['ret_12_1', 'ret_6_1', 'resff3_12_1'],
            'profitability': ['gp_at', 'ni_at', 'ope_be', 'f_score'],
            'investment': ['at_gr1', 'capx_gr1', 'oaccruals_at', 'netis_at'],
            'volatility': ['ivol_capm_252d', 'beta_60m'],
            'liquidity': ['ami_126d', 'turnover_126d', 'zero_trades_126d'],
            'quality': ['qmj', 'qmj_prof', 'qmj_safety'],
            'intangibles': ['rd_me', 'age', 'tangibility']
        },
        'basic': {
            # Simple 4-factor model
            'returns': ['ret_exc'],
            'size': ['market_equity'],
            'value': ['be_me'],
            'momentum': ['ret_12_1']
        }
    }
    
    def __init__(self, wrds_connection):
        self.wrds_db = wrds_connection
        
    def get_available_characteristics(self) -> List[str]:
        """Get all available characteristics from the JKP dataset"""
        schema = self.wrds_db.get_table(library="contrib", table="global_factor", obs=0)
        return schema.columns.tolist()
    
    def validate_characteristics(self, characteristics: List[str]) -> List[str]:
        """Validate that requested characteristics exist in the dataset"""
        available_chars = self.get_available_characteristics()
        valid_chars = []
        invalid_chars = []
        
        for char in characteristics:
            if char in available_chars:
                valid_chars.append(char)
            else:
                invalid_chars.append(char)
        
        if invalid_chars:
            print(f"Warning: Invalid characteristics removed: {invalid_chars}")
            print(f"Available similar characteristics: {[c for c in available_chars if any(inv in c for inv in invalid_chars)]}")
        
        return valid_chars
    
    def load_characteristics_data(self, 
                                factor_model: str = 'fama_french_5',
                                custom_factors: Optional[List[str]] = None,
                                months_back: int = 12,
                                countries: List[str] = ['USA'],
                                filters: Optional[Dict] = None) -> pd.DataFrame:
        """
        Load characteristics data with flexible parameters
        
        Parameters:
        -----------
        factor_model : str
            Predefined factor model ('fama_french_5', 'fama_french_5_extended', 'quality_comprehensive', 
            'trading_frictions', 'academic_research', 'basic')
        custom_factors : List[str], optional
            Custom list of factor names to load
        months_back : int
            Number of months of historical data to load
        countries : List[str]
            List of countries to include
        filters : Dict, optional
            Additional filters for data quality
        """
        
        # Determine which characteristics to load
        if custom_factors:
            characteristics = custom_factors
        elif factor_model in self.FACTOR_MODELS:
            # Flatten all factors in the model
            characteristics = []
            for factor_group in self.FACTOR_MODELS[factor_model].values():
                characteristics.extend(factor_group)
            # Remove duplicates
            characteristics = list(set(characteristics))
        else:
            raise ValueError(f"Unknown factor model: {factor_model}. Available models: {list(self.FACTOR_MODELS.keys())}")
        
        # Validate characteristics exist in dataset
        characteristics = self.validate_characteristics(characteristics)
        
        if not characteristics:
            raise ValueError("No valid characteristics found!")
        
        # Calculate date range
        end_date = datetime.now()
        start_date = end_date - timedelta(days=months_back * 30)  # Approximate
        
        # Default filters for data quality (based on JKP recommendations)
        default_filters = {
            'common': 1,        # Common stocks only
            'exch_main': 1,     # Main exchange only
            'primary_sec': 1,   # Primary security only
            'obs_main': 1       # Main observation only
        }
        
        if filters:
            default_filters.update(filters)
        
        # Build the SQL query
        filter_conditions = []
        for key, value in default_filters.items():
            filter_conditions.append(f"{key}={value}")
        
        # Add country filter
        if countries:
            country_list = "', '".join(countries)
            filter_conditions.append(f"excntry IN ('{country_list}')")
        
        # Add date filter
        filter_conditions.append(f"eom >= '{start_date.strftime('%Y-%m-%d')}'")
        
        # Build characteristics string
        char_string = ', '.join(characteristics)
        
        sql_query = f"""
        SELECT id, eom, gvkey, permno, size_grp, {char_string}
        FROM contrib.global_factor
        WHERE {' AND '.join(filter_conditions)}
        ORDER BY eom DESC
        """
        
        print(f"Loading {len(characteristics)} characteristics for last {months_back} months...")
        print(f"Factor model: {factor_model}")
        print(f"Characteristics: {characteristics}")
        
        return self.wrds_db.raw_sql(sql_query)
    
    def get_factor_model_info(self, model_name: str) -> Dict:
        """Get information about a specific factor model"""
        if model_name in self.FACTOR_MODELS:
            return self.FACTOR_MODELS[model_name]
        else:
            return {}
    
    def list_available_models(self) -> List[str]:
        """List all available factor models"""
        return list(self.FACTOR_MODELS.keys())
    
    def print_model_summary(self, model_name: str):
        """Print detailed summary of a factor model"""
        if model_name not in self.FACTOR_MODELS:
            print(f"Model '{model_name}' not found. Available models: {self.list_available_models()}")
            return
        
        model = self.FACTOR_MODELS[model_name]
        print(f"\n{model_name.upper()} FACTOR MODEL")
        print("=" * 50)
        
        total_chars = 0
        for factor_group, characteristics in model.items():
            print(f"\n{factor_group.upper()}:")
            for char in characteristics:
                print(f"  • {char}")
            total_chars += len(characteristics)
        
        print(f"\nTotal characteristics: {total_chars}")
        print(f"Factor groups: {len(model)}")

# Initialize the data loader
data_loader = JKPDataLoader(wrds_db)

print("📊 JKP Data Loader Initialized")
print("=" * 40)

print("\nAvailable factor models:")
for model in data_loader.list_available_models():
    print(f"  📈 {model}")

# Print detailed summary of the Fama-French 5 model
data_loader.print_model_summary('fama_french_5')

# Load validated Fama-French 5-factor model data
print(f"\n🔄 Loading data...")
usa_data = data_loader.load_characteristics_data(
    factor_model='fama_french_5',
    months_back=12,
    countries=['USA']
)

print(f"\n✅ Successfully loaded {len(usa_data)} observations")
print(f"📅 Date range: {usa_data['eom'].min()} to {usa_data['eom'].max()}")
print(f"🏢 Unique companies: {usa_data['gvkey'].nunique()}")
print(f"📊 Data shape: {usa_data.shape}")

usa_data.head()

# Fama-French 5 Factor Model Characteristics

## Value Factor Characteristics
- **be_me**: Book equity to market equity ratio (classic book-to-market)
- **at_me**: Total assets to market equity ratio
- **sale_me**: Sales to market equity ratio
- **ni_me**: Net income to market equity ratio  
- **ocf_me**: Operating cash flow to market equity ratio
- **ebitda_me**: EBITDA to market equity ratio

## CAPM/Market Factor Characteristics  
- **ret_exc**: Excess returns (return minus risk-free rate)
- **beta_60m**: 60-month market beta
- **beta_252d**: 252-day (1-year) market beta

## Size Factor Characteristics
- **me**: Market equity (shares outstanding × price)
- **market_equity**: Total market value of equity
- **sale_me**: Sales-to-market equity (revenue efficiency)

## Momentum Factor Characteristics
- **ret_12_1**: 12-month return skipping most recent month
- **ret_6_1**: 6-month return skipping most recent month  
- **ret_12_7**: 12-month return skipping most recent 7 months
- **ret_9_1**: 9-month return skipping most recent month

## Quality/Profitability Factor Characteristics
- **ni_at**: Net income to total assets (ROA)
- **ni_be**: Net income to book equity (ROE)
- **roe_be_std**: ROE standard deviation (earnings stability)
- **ocf_at**: Operating cash flow to total assets
- **gp_at**: Gross profits to total assets
- **f_score**: Piotroski F-Score (composite quality measure)

## Additional Factor Models Available
- **profitability**: Focus on earnings, cash flow, margins, returns
- **investment**: Asset growth, capex, financing measures  
- **risk**: Volatility, beta, skewness, maximum return measures
- **basic**: Simple 4-factor model (returns, size, value, momentum)

# Writing onto Influx

In [ ]:
# InfluxDB Setup and Data Processing
# ==================================
# 
# For detailed InfluxDB Enterprise Home setup instructions, 
# see: InfluxDB_Setup_README.md in the project root directory

import numpy as np

# 🔗 Create InfluxDB Client
# =========================
print("🔗 Creating InfluxDB client...")

try:
    # Create InfluxDB client using the configuration
    client = InfluxDBClient3(
        token=influx_config['token'],
        host=influx_config['host'],
        database=influx_config['database'],
        org=influx_config['org']
    )
    
    print(f"✅ InfluxDB client created successfully")
    print(f"🎯 Connected to: {influx_config['host']}")
    print(f"🗄️  Database: {influx_config['database']}")
    
except Exception as e:
    print(f"❌ Failed to create InfluxDB client: {e}")
    client = None

# Improved data processing for InfluxDB
def prepare_data_for_influxdb(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare data for optimal InfluxDB storage
    """
    processed_data = df.copy().set_index("eom")
    
    # Handle missing values appropriately for time series
    processed_data["size_grp"] = processed_data["size_grp"].fillna("unknown")
    processed_data["permno"] = processed_data["permno"].astype("string").fillna("unknown")
    processed_data["gvkey"] = processed_data["gvkey"].fillna("unknown")
    
    # Convert numeric columns to appropriate types
    numeric_columns = processed_data.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        processed_data[col] = pd.to_numeric(processed_data[col], errors='coerce')
    
    # Add metadata columns for better querying
    processed_data['data_vintage'] = datetime.now().strftime('%Y-%m-%d')
    processed_data['factor_model'] = 'fama_french_5'
    
    return processed_data

# Process the data
processed_data = prepare_data_for_influxdb(usa_data)
print(f"Processed data shape: {processed_data.shape}")
processed_data.head()

In [ ]:
class DockerInfluxDBManager:
    """
    🐳 Docker-optimized InfluxDB Manager for JKP factor data
    Enhanced with environment detection and better error handling
    """
    
    def __init__(self, client, config):
        self.client = client
        self.config = config
        self.connected = client is not None
        self.environment = "Docker" if is_docker_environment() else "Local"
        
        print(f"🎯 InfluxDBManager initialized for {self.environment} environment")
        if not self.connected:
            print("⚠️  Manager initialized without InfluxDB connection")
    
    def test_connection(self):
        """Test InfluxDB connection with detailed feedback"""
        if not self.connected:
            print("❌ No InfluxDB client available")
            return False
            
        try:
            # Test connection with a simple query that checks if database exists
            # Use SHOW MEASUREMENTS to see if we can connect and query
            test_query = "SHOW MEASUREMENTS LIMIT 1"
            result = self.client.query(test_query)
            print("✅ InfluxDB connection test successful")
            return True
        except Exception as e:
            error_msg = str(e).lower()
            if any(keyword in error_msg for keyword in ["measurement not found", "no measurements", "empty", "database"]):
                print("✅ InfluxDB connection successful (no measurements found yet - database is empty)")
                return True
            elif "unauthorized" in error_msg or "authentication" in error_msg:
                print(f"❌ InfluxDB authentication failed: {e}")
                return False
            elif "connection" in error_msg or "network" in error_msg:
                print(f"❌ InfluxDB network connection failed: {e}")
                return False
            else:
                print(f"⚠️ InfluxDB test query failed, but connection might be OK: {e}")
                # Try a simpler test - just instantiate a query
                try:
                    # Alternative test: try to access the client's query method
                    if hasattr(self.client, 'query'):
                        print("✅ InfluxDB client is properly initialized")
                        return True
                    else:
                        print("❌ InfluxDB client missing query method")
                        return False
                except Exception as e2:
                    print(f"❌ InfluxDB client validation failed: {e2}")
                    return False
    
    def write_characteristics_data(self, df: pd.DataFrame, measurement_name: str = "jkp_characteristics"):
        """
        Write characteristics data to InfluxDB with Docker-optimized settings
        """
        if not self.connected:
            print("❌ No InfluxDB connection available")
            return False
            
        try:
            print(f"📤 Writing {len(df)} records to measurement: {measurement_name}")
            print(f"🐳 Environment: {self.environment}")
            print(f"🎯 Target: {self.config['host']}")
            
            # Define tag columns (categorical data for efficient querying)
            potential_tags = ["size_grp", "gvkey", "permno", "factor_model", "data_source"]
            tag_columns = [col for col in potential_tags if col in df.columns]
            
            print(f"🏷️  Using tag columns: {tag_columns}")
            print(f"📊 DataFrame shape: {df.shape}")
            
            # Write to InfluxDB with progress feedback
            self.client.write(
                df, 
                data_frame_measurement_name=measurement_name,
                data_frame_tag_columns=tag_columns
            )
            
            print(f"✅ Successfully wrote {len(df)} records to {measurement_name}")
            return True
            
        except Exception as e:
            print(f"❌ Error writing to InfluxDB: {e}")
            print(f"🔧 DataFrame info:")
            print(f"   - Shape: {df.shape}")
            print(f"   - Columns: {list(df.columns)[:10]}..." if len(df.columns) > 10 else f"   - Columns: {list(df.columns)}")
            print(f"   - Index type: {type(df.index)}")
            return False
    
    def write_dataframe(self, df: pd.DataFrame, measurement: str, time_column: str = '_time', tag_columns: List[str] = None):
        """
        Enhanced dataframe writing method with flexible column handling
        """
        if not self.connected:
            print("❌ No InfluxDB connection available")
            return False
            
        try:
            # Prepare dataframe for writing
            df_to_write = df.copy()
            
            # Ensure time column is datetime
            if time_column in df_to_write.columns:
                df_to_write[time_column] = pd.to_datetime(df_to_write[time_column])
                # Set as index if not already
                if df_to_write.index.name != time_column:
                    df_to_write = df_to_write.set_index(time_column)
            
            # Handle tag columns - only use existing columns
            if tag_columns:
                existing_tag_columns = [col for col in tag_columns if col in df_to_write.columns]
            else:
                existing_tag_columns = []
            
            # Write to InfluxDB
            self.client.write(
                df_to_write,
                data_frame_measurement_name=measurement,
                data_frame_tag_columns=existing_tag_columns
            )
            
            return True
            
        except Exception as e:
            print(f"❌ Error in write_dataframe: {e}")
            return False
    
    def query_data(self, query: str):
        """Query data from InfluxDB with enhanced error handling"""
        if not self.connected:
            print("❌ No InfluxDB connection available")
            return None
            
        try:
            print(f"🔍 Executing query on {self.environment} environment...")
            print(f"📋 Query: {query[:100]}{'...' if len(query) > 100 else ''}")
            
            result = self.client.query(query)
            
            if result is not None:
                try:
                    # Try to convert to pandas DataFrame for easier handling
                    if hasattr(result, 'to_pandas'):
                        df = result.to_pandas()
                        print(f"✅ Query successful: {len(df)} records returned")
                        return result
                    else:
                        print("✅ Query successful (non-pandas result)")
                        return result
                except Exception as convert_error:
                    print(f"✅ Query successful, pandas conversion failed: {convert_error}")
                    return result
            else:
                print("⚠️  Query returned no results")
                return None
                
        except Exception as e:
            print(f"❌ Query failed: {e}")
            if "measurement not found" in str(e).lower():
                print("💡 Tip: Make sure you've written data to InfluxDB first")
            return None
    
    def get_factor_data(self, 
                       gvkeys: List[str] = None, 
                       start_date: str = None, 
                       end_date: str = None,
                       factors: List[str] = None,
                       limit: int = 1000):
        """
        🎯 Enhanced convenience method to query specific factor data
        """
        print(f"🔍 Querying factor data with filters:")
        if gvkeys:
            print(f"   📊 Companies: {len(gvkeys)} GVKEYs")
        if start_date:
            print(f"   📅 Start Date: {start_date}")
        if end_date:
            print(f"   📅 End Date: {end_date}")
        if factors:
            print(f"   📈 Factors: {factors[:5]}{'...' if len(factors) > 5 else ''}")
        
        # Build query conditions
        conditions = []
        if gvkeys:
            gvkey_list = "', '".join(gvkeys[:10])  # Limit for performance
            conditions.append(f"gvkey IN ('{gvkey_list}')")
        
        if start_date:
            conditions.append(f"time >= '{start_date}'")
        
        if end_date:
            conditions.append(f"time <= '{end_date}'")
        
        where_clause = " AND ".join(conditions) if conditions else "true"
        
        # Build field selection
        field_selection = "*"
        if factors:
            # Include essential columns
            essential_cols = ["time", "gvkey", "permno"]
            all_fields = essential_cols + [f for f in factors if f not in essential_cols]
            field_selection = ", ".join(all_fields)
        
        query = f"""
        SELECT {field_selection}
        FROM jkp_characteristics
        WHERE {where_clause}
        ORDER BY time DESC
        LIMIT {limit}
        """
        
        return self.query_data(query)
    
    def get_database_summary(self):
        """Get comprehensive database information"""
        if not self.connected:
            print("❌ No InfluxDB connection available")
            return
            
        print(f"📊 InfluxDB Database Summary ({self.environment} Environment)")
        print("=" * 60)
        print(f"🎯 Host: {self.config['host']}")
        print(f"🗄️  Database: {self.config['database']}")
        print(f"🏢 Organization: {self.config['org']}")
        
        # Try to get measurement list
        try:
            measurements_query = "SHOW MEASUREMENTS"
            measurements_result = self.client.query(measurements_query)
            if measurements_result and hasattr(measurements_result, 'to_pandas'):
                measurements_df = measurements_result.to_pandas()
                if not measurements_df.empty:
                    print(f"📈 Available Measurements: {list(measurements_df.iloc[:, 0])}")
                else:
                    print(f"📈 Available Measurements: None (empty database)")
            else:
                print(f"📈 Available Measurements: Unable to retrieve")
        except Exception as e:
            print(f"📈 Available Measurements: Unable to determine ({e})")
        
        # Try to get record count for known measurement
        try:
            count_query = "SELECT COUNT(*) FROM jkp_characteristics"
            count_result = self.client.query(count_query)
            if count_result:
                print(f"📊 Records in jkp_characteristics: Available")
            else:
                print(f"📊 Records in jkp_characteristics: 0 or measurement doesn't exist")
        except Exception as e:
            print(f"📊 Records in jkp_characteristics: Unable to determine ({e})")

# Initialize enhanced InfluxDB manager
print("\n🚀 Initializing InfluxDB Manager...")

if client is not None:
    influx_manager = DockerInfluxDBManager(client, influx_config)
    
    # Test the connection
    print("\n🔍 Testing InfluxDB connection...")
    connection_ok = influx_manager.test_connection()
    
    if connection_ok:
        print("\n📊 Getting database information...")
        influx_manager.get_database_summary()
    else:
        print("\n🔧 Connection troubleshooting:")
        if is_docker_environment():
            print("   Docker Environment:")
            print("   1. Check container status: docker-compose ps")
            print("   2. Check InfluxDB logs: docker-compose logs influxdb")
            print("   3. Wait for InfluxDB startup (can take 60+ seconds)")
            print("   4. Verify from container: ping influxdb")
        else:
            print("   Local Environment:")
            print("   1. Start Docker services: docker-compose up -d")
            print("   2. Check InfluxDB UI: http://localhost:8086")
            print("   3. Verify Docker containers: docker ps")
else:
    print("❌ Could not initialize InfluxDB manager - no client available")
    print("💡 Make sure InfluxDB is running and accessible")
    influx_manager = None

In [ ]:
# DOCKER SETUP VERIFICATION and DATA LOADING
print("✅ Docker environment detected correctly")
print("✅ WRDS connection established") 
print("✅ InfluxDB connection successful")

print(f"\n📊 Data ready to load:")
print(f"   • Records: {len(usa_data):,}")
print(f"   • Companies: {usa_data['gvkey'].nunique():,}")
print(f"   • Date range: {usa_data['eom'].min()} to {usa_data['eom'].max()}")
print(f"   • Columns: {usa_data.shape[1]}")

# Now let's load the data into InfluxDB!
if influx_manager and influx_manager.connected:
    print(f"\n🚀 Loading data into InfluxDB...")
    success = influx_manager.write_characteristics_data(processed_data)
    
    if success:
        print(f"\n🎯 Data loading complete! Verifying...")
        
        # Test query to verify data was loaded
        test_result = influx_manager.query_data("SELECT COUNT(*) FROM jkp_characteristics")
        if test_result:
            print("✅ Data successfully loaded and queryable!")
            
            # Get some sample data
            sample_result = influx_manager.get_factor_data(limit=10)
            if sample_result:
                print("✅ Sample data retrieval successful!")
                
        print(f"\n🎊 SETUP COMPLETE!")
        print("=" * 50)
        print("Your Docker environment is fully operational:")
        print("• ✅ WRDS data loading")
        print("• ✅ InfluxDB storage") 
        print("• ✅ Factor data processing")
        print("• ✅ Query capabilities")
        
        print(f"\n📚 Next steps:")
        print("1. Use influx_manager.get_factor_data() to query data")
        print("2. Access InfluxDB UI at http://localhost:8086")
        print("3. Run factor analysis and portfolio construction")
    else:
        print("❌ Data loading failed - check error messages above")
else:
    print("❌ InfluxDB manager not available")

### Adding Sector Data

In [ ]:
# 🏢 Adding Sector Data to Factor Characteristics
# ===============================================

print("🔄 Loading comprehensive company sector mappings...")

# Load comprehensive gvkey/permno/sector mapping
gvkey_mapping = wrds_db.raw_sql("""
    SELECT
        c.gvkey,
        c.conm,
        c.sic,
        c.gsector,
        c.gind,
        l.lpermno AS permno,
        l.lpermco AS permco,
        s.ticker
    FROM comp.company AS c
    LEFT JOIN crsp.ccmxpf_linktable AS l
        ON c.gvkey = l.gvkey
        AND l.linktype IN ('LU', 'LC')
        AND l.usedflag = 1
    LEFT JOIN crsp.stocknames AS s
        ON l.lpermno = s.permno
    WHERE c.gvkey IS NOT NULL
""")

print(f"✅ Loaded {len(gvkey_mapping)} company mappings")
print(f"📊 Unique GVKEYs: {gvkey_mapping['gvkey'].nunique()}")
print(f"📊 Unique PERMNOs: {gvkey_mapping['permno'].nunique()}")

# Display sample of the mapping data
print(f"\n📋 Sample company mappings:")
gvkey_mapping.head(10)

In [ ]:
# 🧹 Clean and Prepare Sector Mapping Data
# ========================================

print("🧹 Cleaning sector mapping data...")

# Clean the mapping data
gvkey_mapping_clean = gvkey_mapping.copy()

# Handle missing values and data types
gvkey_mapping_clean['gvkey'] = gvkey_mapping_clean['gvkey'].astype(str).str.strip()
# gvkey_mapping_clean['permno'] = gvkey_mapping_clean['permno'].fillna(0).astype('Int64')
# gvkey_mapping_clean['permco'] = gvkey_mapping_clean['permco'].fillna(0).astype('Int64')

# Clean sector codes and names
gvkey_mapping_clean['gsector'] = gvkey_mapping_clean['gsector'].fillna('Unknown')
gvkey_mapping_clean['gind'] = gvkey_mapping_clean['gind'].fillna('Unknown')

# Fix SIC handling - convert to numeric first, then handle missing values
gvkey_mapping_clean['sic'] = pd.to_numeric(gvkey_mapping_clean['sic'], errors='coerce').fillna(0).astype('Int64')

gvkey_mapping_clean['ticker'] = gvkey_mapping_clean['ticker'].fillna('Unknown').astype(str).str.strip()
gvkey_mapping_clean['conm'] = gvkey_mapping_clean['conm'].fillna('Unknown').astype(str).str.strip()

# Remove duplicates - keep most recent record per gvkey
gvkey_mapping_unique = gvkey_mapping_clean.drop_duplicates(subset=['gvkey'], keep='last')

print(f"✅ Cleaned mapping data:")
print(f"   📊 Total records: {len(gvkey_mapping_unique)}")
print(f"   📊 Unique GVKEYs: {gvkey_mapping_unique['gvkey'].nunique()}")
print(f"   📊 Records with sector data: {(gvkey_mapping_unique['gsector'] != 'Unknown').sum()}")
print(f"   📊 Records with SIC codes: {(gvkey_mapping_unique['sic'] > 0).sum()}")
print(f"   📊 Records with tickers: {(gvkey_mapping_unique['ticker'] != 'Unknown').sum()}")

# Display sector distribution
print(f"\n📈 Sector Distribution:")
sector_counts = gvkey_mapping_unique['gsector'].value_counts()
print(sector_counts.head(10))

gvkey_mapping_unique.head()

In [ ]:
# 📊 Query Existing Factor Data from InfluxDB
# ============================================

print("📊 Querying existing factor data from InfluxDB...")

try:
    # Query all factor characteristics data
    query = '''
    from(bucket: "factor_modeling")
        |> range(start: 1990-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
        |> filter(fn: (r) => r["_measurement"] == "jkp_characteristics")
        |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
        |> drop(columns: ["_start", "_stop", "_measurement"])
    '''
    
    # Create a simple client for querying (without callback to avoid issues)
    query_client = InfluxDBClient3(
        host="localhost",
        port=8086,
        database="factor_modeling",
        token=token
    )
    
    print("🔍 Executing query...")
    factor_data_df = query_client.query(query=query, language="flux")
    
    print(f"✅ Successfully retrieved factor data:")
    print(f"   📊 Total records: {len(factor_data_df)}")
    print(f"   📊 Date range: {factor_data_df['_time'].min()} to {factor_data_df['_time'].max()}")
    print(f"   📊 Unique GVKEYs: {factor_data_df['gvkey'].nunique()}")
    print(f"   📊 Columns: {list(factor_data_df.columns)}")
    
    # Show sample of data
    print(f"\n📋 Sample factor data:")
    display(factor_data_df.head())
    
    query_client.close()
    
except Exception as e:
    print(f"❌ Error querying factor data: {e}")
    print("💡 This might be due to Docker networking. Let's try the Docker manager...")
    
    # Alternative: Use the Docker manager we created earlier
    try:
        # Use our Docker manager with proper client setup
        docker_manager = DockerInfluxDBManager(
            url="http://influxdb:8086",
            token=token,
            org="qtab",
            bucket="factor_modeling"
        )
        
        # Simplified query using query API
        print("🔄 Trying alternative query method...")
        simple_query = '''
        from(bucket: "factor_modeling")
            |> range(start: 1990-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
            |> filter(fn: (r) => r["_measurement"] == "jkp_characteristics")
            |> limit(n: 1000)
        '''
        
        result = docker_manager.client.query_api().query_data_frame(simple_query)
        
        if isinstance(result, list) and len(result) > 0:
            factor_data_df = result[0]  # Take first dataframe if list
        else:
            factor_data_df = result
            
        print(f"✅ Alternative query successful:")
        print(f"   📊 Records retrieved: {len(factor_data_df)}")
        display(factor_data_df.head())
        
    except Exception as e2:
        print(f"❌ Alternative query also failed: {e2}")
        print("💡 Will proceed with sector mapping and manual data combination")
        factor_data_df = None

In [ ]:
# 🔗 Join Factor Data with Sector Information
# ===========================================

print("🔗 Joining factor data with sector mappings...")

if factor_data_df is not None and len(factor_data_df) > 0:
    # Ensure gvkey is string for proper joining
    factor_data_df['gvkey'] = factor_data_df['gvkey'].astype(str).str.strip()
    
    # Perform left join to add sector information
    enhanced_factor_data = factor_data_df.merge(
        gvkey_mapping_unique[['gvkey', 'conm', 'ticker', 'sic', 'gsector', 'gind', 'permno', 'permco']],
        on='gvkey',
        how='left'
    )
    
    print(f"✅ Join completed successfully:")
    print(f"   📊 Original factor records: {len(factor_data_df)}")
    print(f"   📊 Enhanced records: {len(enhanced_factor_data)}")
    print(f"   📊 Records with sector data: {(enhanced_factor_data['gsector'].notna() & (enhanced_factor_data['gsector'] != 'Unknown')).sum()}")
    print(f"   📊 Coverage rate: {((enhanced_factor_data['gsector'].notna() & (enhanced_factor_data['gsector'] != 'Unknown')).sum() / len(enhanced_factor_data) * 100):.1f}%")
    
    # Show the new columns added
    new_columns = ['conm', 'ticker', 'sic', 'gsector', 'gind', 'permno', 'permco']
    print(f"\n📊 New columns added: {new_columns}")
    
    # Display sample of enhanced data
    print(f"\n📋 Sample enhanced data with sector information:")
    sample_with_sectors = enhanced_factor_data[enhanced_factor_data['gsector'].notna() & (enhanced_factor_data['gsector'] != 'Unknown')].head()
    display(sample_with_sectors[['_time', 'gvkey', 'conm', 'ticker', 'gsector', 'gind', 'sic', 'mom_12m', 'prof', 'size']].head())
    
    # Sector distribution in the enhanced dataset
    print(f"\n📈 Sector distribution in enhanced factor data:")
    sector_dist = enhanced_factor_data['gsector'].value_counts()
    print(sector_dist.head(10))
    
else:
    print("⚠️ No factor data available for joining. Will create sector mapping as standalone reference.")
    enhanced_factor_data = gvkey_mapping_unique
    print(f"📊 Created sector reference with {len(enhanced_factor_data)} company records")

In [ ]:
# 💾 Write Enhanced Data to InfluxDB
# ==================================

print("💾 Writing enhanced factor data with sector information to InfluxDB...")

try:
    # Use the existing influx_manager or create a clean one
    if 'influx_manager' in locals() and influx_manager and influx_manager.connected:
        write_manager = influx_manager
        print("✅ Using existing InfluxDB manager")
    else:
        # Create a clean client for writing
        print("🔧 Creating clean InfluxDB client for writing...")
        clean_client = InfluxDBClient3(
            token=influx_config['token'],
            host=influx_config['host'],
            database=influx_config['database'],
            org=influx_config['org']
        )
        write_manager = DockerInfluxDBManager(clean_client, influx_config)
    
    if 'enhanced_factor_data' in locals() and len(enhanced_factor_data) > 0:
        
        # Prepare data for InfluxDB writing
        data_to_write = enhanced_factor_data.copy()
        
        # Ensure proper data types for InfluxDB
        if '_time' in data_to_write.columns:
            data_to_write['_time'] = pd.to_datetime(data_to_write['_time'])
            time_col = '_time'
        elif 'eom' in data_to_write.columns:
            data_to_write['eom'] = pd.to_datetime(data_to_write['eom'])
            time_col = 'eom'
        else:
            time_col = None
        
        # Fill NaN values for InfluxDB compatibility
        for col in ['conm', 'ticker', 'gsector', 'gind']:
            if col in data_to_write.columns:
                data_to_write[col] = data_to_write[col].fillna('Unknown')
        
        for col in ['sic', 'permno', 'permco']:
            if col in data_to_write.columns:
                data_to_write[col] = data_to_write[col].fillna(0)
        
        # Convert gvkey to string for consistency
        if 'gvkey' in data_to_write.columns:
            data_to_write['gvkey'] = data_to_write['gvkey'].astype(str)
        
        # Write to a new measurement for enhanced data
        measurement_name = "jkp_characteristics_enhanced"
        
        print(f"📊 Writing {len(data_to_write)} records to measurement: {measurement_name}")
        
        # Write data in batches to avoid memory issues
        batch_size = 1000
        total_batches = (len(data_to_write) + batch_size - 1) // batch_size
        
        successful_batches = 0
        for i in range(0, len(data_to_write), batch_size):
            batch = data_to_write.iloc[i:i + batch_size].copy()
            batch_num = (i // batch_size) + 1
            
            print(f"   📝 Writing batch {batch_num}/{total_batches} ({len(batch)} records)...")
            
            try:
                # Use the standard write method with proper parameters
                tag_columns = []
                if 'gvkey' in batch.columns:
                    tag_columns.append('gvkey')
                if 'gsector' in batch.columns:
                    tag_columns.extend(['gsector', 'gind', 'ticker'])
                
                # Set time column as index if available
                if time_col and time_col in batch.columns:
                    batch = batch.set_index(time_col)
                
                write_manager.client.write(
                    batch,
                    data_frame_measurement_name=measurement_name,
                    data_frame_tag_columns=tag_columns
                )
                
                successful_batches += 1
                print(f"      ✅ Batch {batch_num} written successfully")
                
            except Exception as batch_error:
                print(f"      ❌ Failed to write batch {batch_num}: {batch_error}")
                continue
        
        print(f"\n📊 Write Summary:")
        print(f"   ✅ Successful batches: {successful_batches}/{total_batches}")
        print(f"   📈 Records written: ~{successful_batches * batch_size}")
        
        if successful_batches > 0:
            print(f"\n🎉 Enhanced factor data processing complete!")
            print(f"📊 New measurement created: {measurement_name}")
            print(f"📈 Enhanced data includes company names, tickers, sector classifications, and industry codes")
            print(f"🔧 You can now query this enhanced dataset for sector-specific factor analysis!")
        else:
            print(f"\n❌ No batches were written successfully")
    
    else:
        print("⚠️ No enhanced data available to write")
        
except Exception as e:
    print(f"❌ Error writing enhanced data: {e}")
    import traceback
    traceback.print_exc()

print(f"\n💡 Alternative: Use clean_client for simple writes:")
print(f"   clean_client.write(your_data, data_frame_measurement_name='measurement_name')")

### Data Pipeline:
```
WRDS (JKP Global Factor) → Python Data Loader → InfluxDB → Factor Analysis
     ✅ Connected           ✅ Working         ✅ Stored    🚀 Ready!
```

### Accessing Data
1. **InfluxDB UI**: http://localhost:8086
   - Username: `admin` | Password: `factormodel2024`
   - Organization: `qtab` | Bucket: `jkp_factors`
   - Measurement: `jkp_characteristics`

2. **Sample Query for Data Explorer:**
   ```flux
   from(bucket: "jkp_factors")
     |> range(start: -30d)
     |> filter(fn: (r) => r._measurement == "jkp_characteristics")
     |> filter(fn: (r) => r.gvkey != "")
     |> limit(n: 100)
   ```

### Data Overview
- **Factor Analysis**: All Fama-French 5 factors available
- **Date range**: 2024-08-31 to 2025-02-28
- **Cross-Section**: 5,245 companies across size groups
- **Characteristics**: Value, momentum, profitability, investment, market factors

In [ ]:
# 🔧 Clean InfluxDB Client (No Callback Errors)
# ===============================================

print("🔧 Creating clean InfluxDB client without callbacks...")

# Create a simple client without any callbacks to eliminate errors
try:
    clean_client = InfluxDBClient3(
        token=influx_config['token'],
        host=influx_config['host'],
        database=influx_config['database'],
        org=influx_config['org']
        # No write_client_options = no callback errors!
    )
    
    print(f"✅ Clean InfluxDB client created successfully")
    print(f"🎯 Host: {influx_config['host']}")
    print(f"🗄️  Database: {influx_config['database']}")
    print(f"🏢 Organization: {influx_config['org']}")
    
    # Test write with a small sample (no callback messages)
    print(f"\n🧪 Testing clean write (no callback errors)...")
    test_sample = processed_data.head(5).copy()
    
    clean_client.write(
        test_sample, 
        data_frame_measurement_name="clean_test_characteristics",
        data_frame_tag_columns=["size_grp", "gvkey"]
    )
    
    print(f"✅ Clean write successful - no callback errors!")
    print(f"📊 {len(test_sample)} records written silently")
    
    print(f"\n💡 USE THIS CLIENT FOR ERROR-FREE WRITES:")
    print(f"   clean_client.write(your_data, data_frame_measurement_name='measurement_name')")
    
except Exception as e:
    print(f"❌ Clean client creation failed: {e}")
    clean_client = None

print(f"\n🎯 CALLBACK ERROR EXPLANATION:")
print("=" * 40)
print("• InfluxDB sends 'bytes' data to callback functions")
print("• Our callback tried to use .split() on bytes (needs string)")
print("• This is a cosmetic issue - data writes still work!")
print("• Using clean_client eliminates all callback errors")
print("• Your 34,074 records are safely stored regardless!")

In [ ]:
# Data Granularity Analysis: Beyond Monthly EOM Data
# ===================================================

def analyze_data_granularity():
    """
    Analyze available data frequencies in the JKP dataset and WRDS
    """
    print("JKP Global Factor Dataset Analysis:")
    print("=" * 50)
    
    # Check the date frequency in our current data
    if not usa_data.empty:
        dates = pd.to_datetime(usa_data['eom']).sort_values()
        date_diffs = dates.diff().dropna()
        
        print(f"Current dataset date range: {dates.min()} to {dates.max()}")
        print(f"Number of unique dates: {len(dates.unique())}")
        print(f"Most common date difference: {date_diffs.mode().iloc[0] if not date_diffs.empty else 'N/A'}")
        
        # Analyze frequency
        if len(dates) > 1:
            avg_diff = date_diffs.mean()
            if avg_diff.days < 7:
                frequency = "Daily or higher"
            elif avg_diff.days < 14:
                frequency = "Weekly"
            elif avg_diff.days < 32:
                frequency = "Monthly"
            elif avg_diff.days < 100:
                frequency = "Quarterly"
            else:
                frequency = "Annual or lower"
            
            print(f"Detected frequency: {frequency}")
    
    print("\nData Granularity Options in WRDS:")
    print("-" * 40)
    print("1. JKP Global Factor (contrib.global_factor): Monthly EOM")
    print("2. CRSP Daily Stock File: Daily returns and prices")
    print("3. CRSP Monthly Stock File: Monthly returns")
    print("4. Compustat Daily: Daily fundamental ratios (limited)")
    print("5. Compustat Quarterly: Quarterly fundamentals")
    print("6. TAQ (if available): Intraday tick data")
    
    print("\nRecommendations for Factor Modeling:")
    print("-" * 40)
    print("• Monthly EOM (JKP): Best for factor research, comprehensive characteristics")
    print("• Daily returns (CRSP): For portfolio backtesting and risk management")
    print("• Quarterly fundamentals (Compustat): For custom characteristic construction")
    print("• Intraday (TAQ): For microstructure research (usually not needed for factors)")

def check_higher_frequency_data():
    """
    Check if higher frequency data is available
    """
    print("\nChecking for higher frequency data sources...")
    
    # Check CRSP daily data availability
    try:
        crsp_daily_schema = wrds_db.get_table(library="crsp", table="dsf", obs=0)
        print(f"✓ CRSP Daily Stock File available with {len(crsp_daily_schema.columns)} columns")
        print("  Key columns: date, permno, ret, prc, vol, shrout")
    except:
        print("✗ CRSP Daily Stock File not accessible")
    
    # Check monthly CRSP
    try:
        crsp_monthly_schema = wrds_db.get_table(library="crsp", table="msf", obs=0)
        print(f"✓ CRSP Monthly Stock File available with {len(crsp_monthly_schema.columns)} columns")
    except:
        print("✗ CRSP Monthly Stock File not accessible")
    
    # Check Compustat quarterly
    try:
        comp_quarterly_schema = wrds_db.get_table(library="comp", table="fundq", obs=0)
        print(f"✓ Compustat Quarterly available with {len(comp_quarterly_schema.columns)} columns")
    except:
        print("✗ Compustat Quarterly not accessible")

# Run the analysis
analyze_data_granularity()
check_higher_frequency_data()

## Deprecated

In [ ]:
factor_returns = pd.read_csv("factor-returns.csv")
factor_returns = factor_returns.set_index("date")
factor_returns

In [ ]:
client.write(factor_returns, 
             data_frame_measurement_name="factor_returns", 
             data_frame_tag_columns=["location", "name", "freq","weighting"])

In [ ]:
wrds_db.list_tables(library="comp")

In [ ]:
comp_comp = wrds_db.get_table(library="comp", table="company")
comp_keys = comp_comp.gvkey.unique()

In [ ]:
wrds_db.list_tables(library="crsp")

In [ ]:
comp_crsp_linker = wrds_db.get_table(library="crsp", table="ccmxpf_lnkhist")

In [ ]:
link_keys = comp_crsp_linker.gvkey.unique()

In [ ]:
comp_comp[comp_comp.gvkey=="011271"]

In [ ]:
a = set(comp_keys)
b = set(link_keys)
a-b